In [18]:
import os
os.environ["OPENAI_API_KEY"] = "sk-xfY0F4FKPS6oLPSWC29aF6E5Eb3841E38f79Ef1e8780Ef58"
os.environ["OPENAI_BASE_URL"] = "https://api.ai-gaochao.cn/v1"

In [19]:
import os
import json
from typing import List, Dict

def load_jsonl_files(folder_path: str) -> List[Dict]:
    """
    加载文件夹中所有JSONL文件，返回所有chunk的列表
    :param folder_path: JSONL文件所在文件夹路径
    :return: 包含所有chunk的列表，每个元素是一个chunk字典
    """
    chunks = []
    # 遍历文件夹中的所有文件
    for filename in os.listdir(folder_path):
        if filename.endswith(".jsonl"):  # 只处理JSONL文件
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:  # 跳过空行
                        chunk = json.loads(line)  # 解析JSON行
                        chunks.append(chunk)
    print(f"成功加载 {len(chunks)} 个chunk")
    return chunks

# 示例调用：加载EEjsonl文件夹中的所有chunk
folder_path = "/data1/oysl/RAG/jsons_EE_Chi"  # 你的JSONL文件夹路径
all_chunks = load_jsonl_files(folder_path)

成功加载 555 个chunk


In [20]:
# 检查chunk的实际结构
print("📊 检查chunk结构...")
if all_chunks:
    sample_chunk = all_chunks[0]
    print(f"第一个chunk的键: {list(sample_chunk.keys())}")
    print(f"第一个chunk的内容示例:")
    for key, value in sample_chunk.items():
        if isinstance(value, str):
            print(f"  {key}: {value[:100]}...")
        elif isinstance(value, list):
            print(f"  {key}: {value[:3]}...")
        else:
            print(f"  {key}: {value}")
    print("\n" + "="*50)

📊 检查chunk结构...
第一个chunk的键: ['chunk_id', '公式', '完整上下文', '变量定义', '关键词', '潜在问题', 'task_description', '所属章节', '难度级别', '学科领域']
第一个chunk的内容示例:
  chunk_id: DL_T_5044-2014-A.3.1-001...
  公式: I_{\mathrm{n}} \geq K_{\mathrm{k}} I_{\mathrm{m}}...
  完整上下文: 充电装置输出回路断路器额定电流应按充电装置额定输出电流选择，计算公式为：I_n ≥ K_k I_m。该公式用于确定直流电源系统中充电装置输出回路断路器的额定电流，确保其满足负载要求并考虑可靠系数。...
  变量定义: [{'符号': 'I_n', '名称': '断路器额定电流', '物理意义': '回路保护电器的额定电流值', '单位': 'A'}, {'符号': 'K_k', '名称': '可靠系数', '物理意义': '考虑计算误差和可靠性的安全系数', '单位': '无量纲（取1.2）'}, {'符号': 'I_m', '名称': '充电装置额定输出电流', '物理意义': '充电装置持续输出的最大工作电流', '单位': 'A'}]...
  关键词: ['断路器', '额定电流', '充电装置']...
  潜在问题: ['已知充电装置额定输出电流I_m=100A，求断路器最小额定电流I_n']...
  task_description: 1.获取充电装置额定输出电流I_m；2.选取可靠系数K_k=1.2；3.计算I_n=K_k×I_m；4.向上取整至标准断路器额定电流值...
  所属章节: 附录A.3.1...
  难度级别: 初级...
  学科领域: 电力工程...



In [21]:
# 文本清理与日志工具
import re
from typing import Any

def _strip_latex_markdown(text: Any) -> str:
    r"""
    清理文本中的 LaTeX/Markdown 痕迹与零宽字符，尽量保留语义内容。
    - 去除：$, $$, \( \), \[ \], \left \right 等定界符
    - 展平：\text{...}、\mathrm{...} 等命令，保留花括号内容
    - 替换：\times -> ×
    - 清理：零宽字符与多余反斜杠
    """
    if not isinstance(text, str):
        return str(text)
    s = text
    # 去零宽字符
    s = re.sub(r"[\u200b\u200c\u200d]", "", s)
    # 去美元符与行间公式定界
    s = re.sub(r"\$\$?", "", s)
    # 去行内/行间公式括号
    s = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", s)
    # 去 \left \right 等尺寸控制
    s = re.sub(r"\\left|\\right", "", s)
    # 展平 \text{...} / \mathrm{...} / \mathbf{...} / \mathit{...}
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\mathrm\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\mathbf\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\mathit\{([^}]*)\}", r"\1", s)
    # 常见命令替换
    s = re.sub(r"\\times", "×", s)
    s = re.sub(r"\\cdot", "·", s)
    # 剩余命令名去除（保留其后普通字符）
    s = re.sub(r"\\[a-zA-Z]+", "", s)
    # 多余空白合并
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 相似度匹配日志缓冲区（避免Notebook折叠后看不到详细过程）
try:
    _similarity_log_buffer  # type: ignore[name-defined]
except NameError:
    _similarity_log_buffer = []

In [22]:
## GPT-4o -> DeepSeek-V3 API配置和复合题目拆解
from openai import OpenAI
import os
import re
from typing import List, Dict, Tuple

# 配置硅基流动 DeepSeek-V3 API客户端
_api_key = "sk-sdjnmoxobkargejcuarscimjxwhtfuwmglcurrgcgfkjaonv"
_base_url = "https://api.siliconflow.cn/v1"
client = OpenAI(api_key=_api_key, base_url=_base_url)

def extract_sub_problems(text: str) -> List[str]:
    """从拆解文本中提取子问题列表，若解析失败则回退为整题。"""
    if not isinstance(text, str) or not text.strip():
        return []
    
    # 稍微清理空行
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    subs: List[str] = []
    
    # --- 逻辑 1：按“子问题”拆分 ---
    # 支持格式：【子问题1】、子问题1：、**子问题1**、## 子问题1 等
    header_regex = re.compile(r"^[\*\#]*[【\[]?子问题\s*\d+")
    
    current_content = []
    found_first_header = False
    
    for ln in lines:
        if header_regex.match(ln):
            # 遇到新标题
            if found_first_header:
                # 保存前一个缓冲区
                full_text = " ".join(current_content).strip()
                if full_text:
                    subs.append(full_text)
                current_content = []
            
            found_first_header = True
            # 去掉标题本身，保留该行后续内容
            # 先去前缀
            content_part = re.sub(r"^[\*\#]*[【\[]?子问题\s*\d+[】\]]?[:：]?", "", ln).strip()
            if content_part:
                current_content.append(content_part)
        else:
            # 普通行
            if found_first_header:
                current_content.append(ln)
                
    # 循环结束后，保存最后一个缓冲区
    if found_first_header:
        full_text = " ".join(current_content).strip()
        if full_text:
            subs.append(full_text)

    # --- 逻辑 2：按纯数字编号拆分 (备用) ---
    # 仅当逻辑1完全没匹配到时启用 且 文本看起来像列表
    if not subs:
        current_content = []
        found_first_header = False
        number_regex = re.compile(r"^[\*\#]*\d+[\.、]")
        
        for ln in lines:
            if number_regex.match(ln):
                if found_first_header:
                    full_text = " ".join(current_content).strip()
                    if full_text:
                        subs.append(full_text)
                    current_content = []
                
                found_first_header = True
                content_part = re.sub(r"^[\*\#]*\d+[\.、]\s*", "", ln).strip()
                if content_part:
                    current_content.append(content_part)
            else:
                if found_first_header:
                    current_content.append(ln)
                    
        if found_first_header:
            full_text = " ".join(current_content).strip()
            if full_text:
                subs.append(full_text)
    
    # --- 逻辑 3：保底 ---
    # 如果两种方式都没提取到，或者提取出了空列表，说明文本本身不是分步结构
    if not subs and text.strip():
        subs = [text.strip()]
        
    return subs

def decompose_complex_problem(user_question: str) -> List[str]:
    """使用DeepSeek-V3 API拆解复合题目为多个子问题（示例驱动版）。"""

    decompose_prompt = f"""
你是一位思维严谨的工程领域问题拆解专家。你的任务是将复合问题分解为一系列逻辑严密、顺序正确的子问题。每个子问题都应是可独立解决的最小单元，且前后步骤的输入输出必须连贯。

【拆解核心原则】
1. 逻辑链优先：从目标量开始，反向推导构建完整的变量依赖链
2. 最小化与连贯性：每个子问题只解决一个核心计算，只代表一个公式(如果你发现某个公式中需要用到的量无法直接从题干中获得，就说明这个公式之前还需要别的子问题，这个用到的量要通过前面的子问题求得)
3. 专业领域识别：准确识别工程中的温度修正、盐密计算等专门流程
4. 子问题数量不固定：请根据实际变量依赖链自动拆解，可能是2步、3步、4步或更多（即有可能是两个子问题，三个子问题，四个子问题或更多）

【工程领域真实示例】

问题（工程）："某变电站运维人员对一片污秽绝缘子进行清洗试验，以评估其污秽程度。实测清洗液在温度 θ = 28°C 时的体积电导率 σ_θ = 0.15 S/m，温度系数 b = 0.032/°C。清洗时使用了 V = 800 cm³ 的蒸馏水，绝缘子的有效表面积 A = 1200 cm²。请计算该绝缘子的等值盐密（ESDD）。"

拆解思维：
- 首先，使用电导率温度校正公式，将实测电导率校正到标准温度 20°C 下的值 σ₂₀
- 然后，使用校正后的电导率 σ₂₀ 计算盐度 S_s
- 最后，使用盐度 S_s 和清洗参数计算等值盐密 ESDD

拆解输出：

【子问题1】已知实测电导率 σ_θ = 0.15 S/m、温度系数 b = 0.032/°C、温度 θ = 28°C，根据温度校正公式 σ₂₀ = σ_θ × [1 - b(θ - 20)]，求 20°C 时的电导率 σ₂₀。

【子问题2】已知 σ₂₀，根据盐度公式 S_s = (5.7 × σ₂₀)^1.03，求盐度 S_s。

【子问题3】已知 S_s、V = 800 cm³、A = 1200 cm²，根据等值盐密公式 ESDD = S_s × V / A，求等值盐密 ESDD。

【示例B：导体温升与损耗（2个子问题）】

问题（工程）："某铜导体在20°C时的电阻 R20 = 0.50 Ω，电阻温度系数 α = 0.004/°C，工作电流 I = 50 A，工作温度 θ = 60°C。求在工作温度下的功率损耗 P_loss。"

拆解输出：
【子问题1】根据温度修正公式 R_θ = R_20 × [1 + α(θ - 20)]，计算工作温度下的电阻 R_θ。

【子问题2】根据损耗公式 P_loss = I^2 × R_θ，计算功率损耗 P_loss。

【示例C：三相线路压降与损耗（4个子问题）】

问题（工程）："三相对称供电系统，额定线电压 U_N = 10 kV，线路长度 L = 5 km，单位长度电阻 r = 0.3 Ω/km，单位长度电抗 x = 0.2 Ω/km，负载电流 I = 120 A，功率因数 cosφ = 0.85（滞后）。求线路压降和有功损耗。"

拆解输出：
【子问题1】计算线路总电阻 R = r × L。

【子问题2】计算线路总电抗 X = x × L。

【子问题3】计算线路压降 ΔU = √3 × I × (R × cosφ + X × sinφ)。

【子问题4】计算有功损耗 P_loss = 3 × I^2 × R。

你的任务
请严格遵循以上拆解思维，为以下问题构建最优求解路径：

【问题复述】
{user_question}

【输出格式要求】

【变量提取】
已知变量: [列出所有已知量及其数值]
目标变量: [最终要求的量]

【公式路径分析】
请根据实际变量依赖链，逐步列出每一步所用公式、输入变量和输出变量

【子问题拆解】
请按实际公式链自动拆解，子问题数量不固定（可能是2步、3步、4步或更多）

【子问题N】 [详细描述第N个计算步骤，包含具体数值和公式名称]
→ 输入: [变量名] | 公式: [具体公式] | 输出: [结果变量]

【格式要求补充】
- 所有变量和公式必须使用标准的LaTeX格式（例如：使用 \\sigma_\\theta 而不是 σ_θ，使用 \\times 而不是 ×）
- 不要使用Markdown公式定界符（如 $ 或 $$），直接写LaTeX代码即可
- 输出内容务必清晰、可解析
"""

    try:
        print("🔧 正在调用DeepSeek-V3拆解复合题目...")
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3",
            messages=[
                {"role": "system", "content": "你是一个专业的工程领域问题分析专家，擅长将复杂的工程计算问题拆解为多个逻辑相关的子问题。"},
                {"role": "user", "content": decompose_prompt}
            ],
            temperature=0.1,
            max_tokens=2000
        )
        decomposed_text = response.choices[0].message.content or ""
        
        # 仅去除代码块标记，保留换行符以支持分步解析
        # 也不要使用 _strip_latex_markdown，因为它会移除关键的 LaTeX 命令（如 \sigma, \times）并合并换行
        decomposed_text_clean = re.sub(r"^```(json|markdown)?|```$", "", decomposed_text.strip(), flags=re.MULTILINE)
        
        print("✅ DeepSeek-V3拆解完成")
        print(f"📋 拆解结果（原始）：\n{decomposed_text}")
        print(f"📋 拆解结果（保留格式）：\n{decomposed_text_clean}")
        
        sub_problems = extract_sub_problems(decomposed_text_clean)
        print(f"🔍 提取到 {len(sub_problems)} 个子问题")
        return sub_problems
    except Exception as e:
        print(f"❌ DeepSeek-V3 API调用失败: {str(e)}")
        print("🔄 回退到原始问题处理")
        return [user_question] if user_question.strip() else []

In [23]:
from transformers import AutoTokenizer, AutoModel
import json
import torch
import os
import gc
from pathlib import Path

# 缓存配置
EMB_CACHE_DIR = "emb_cache_chi"
EMB_CACHE_FILE = os.path.join(EMB_CACHE_DIR, "chunks_embeddings.pt")
EMB_META_FILE = os.path.join(EMB_CACHE_DIR, "chunks_meta.json")

# 全局缓存变量
_cached_chunk_embs = None
_cached_chunk_ids = None

def load_embedding_components(model_path: str = "/data1/oysl/RAG/Qwen3-Embedding") -> Tuple[AutoModel, AutoTokenizer]:
    """加载Qwen3-Embedding模型与分词器。"""
    print("🔄 正在初始化模型 (强制 CPU 模式)...")
    
    # 彻底清理显存，防止之前的碎片干扰
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # 【修复】修正 map_location 参数传递错误
    # map_location 不是 from_pretrained 的直接参数，应该通过 device_map 或者 torch.load 的机制
    # 对于纯 CPU 加载，最稳妥的方式是使用 device_map="cpu"
    model = AutoModel.from_pretrained(
        model_path, 
        trust_remote_code=True, 
        device_map="cpu",  # 使用 device_map="cpu" 替代 map_location
    )
    model.eval()
    return model, tokenizer

def get_embeddings(texts: List[str], model, tokenizer, device: str = None) -> torch.Tensor:
    """
    对文本列表生成L2归一化的句向量。
    注意：鉴于显存受限，建议默认使用 CPU 进行 Embedding 推理。
    """
    # 如果外部未指定 device，则默认使用 model 所在的 device
    if device is None:
        device = str(model.device) if hasattr(model, "device") else "cpu"
        
    target_device = device

    # 准备输入
    # 注意：确保输入张量与模型在同一设备
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    enc = {k: v.to(target_device) for k, v in enc.items()}
    
    # 确保模型也在目标设备
    current_device = str(model.device)
    if target_device != "cpu" and current_device != target_device:
        try:
            model.to(target_device)
        except Exception as e:
            print(f"⚠️ 无法将模型移动到 {target_device}: {e}。保持在 {current_device} 运行。")
            target_device = current_device
            enc = {k: v.to(target_device) for k, v in enc.items()}

    with torch.no_grad():
        try:
            out = model(**enc)
        except RuntimeError as e:
            # 捕获设备不一致错误 (如 input 在 cuda, weight 在 cpu)
            if "device" in str(e).lower() and target_device != "cpu":
                 print(f"⚠️ 检测到设备冲突 ({e})，正在紧急回退到 CPU...")
                 model.to("cpu")
                 enc = {k: v.to("cpu") for k, v in enc.items()}
                 out = model(**enc)
            else:
                raise e

        emb = out.last_hidden_state.mean(dim=1)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        
    return emb.cpu()

def precompute_chunk_embeddings(chunks: List[Dict], model, tokenizer) -> None:
    """为知识库预计算并缓存向量。"""
    global _cached_chunk_embs, _cached_chunk_ids
    Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    texts, chunk_ids = [], []
    for idx, ch in enumerate(chunks):
        if not isinstance(ch, dict):
            continue
        desc = ch.get("task_description") or ch.get("公式") or ch.get("标题") or ch.get("完整上下文") or f"chunk_{idx}"
        desc = _strip_latex_markdown(str(desc))
        texts.append(desc)
        chunk_ids.append(str(ch.get("chunk_id", f"idx_{idx}")))
    if not texts:
        _cached_chunk_embs, _cached_chunk_ids = None, None
        return
    
    print("⚠️ 正在生成 Embeddings (使用 CPU 避免 OOM)... 这可能需要一点时间")
    # 强制传递 device="cpu"
    embs = get_embeddings(texts, model, tokenizer, device="cpu")
    
    _cached_chunk_embs = embs
    _cached_chunk_ids = chunk_ids
    torch.save(embs, EMB_CACHE_FILE)
    with open(EMB_META_FILE, "w", encoding="utf-8") as f:
        json.dump(chunk_ids, f, ensure_ascii=False)

def _load_cached_embeddings() -> bool:
    """尝试从磁盘加载缓存。成功返回True。"""
    global _cached_chunk_embs, _cached_chunk_ids
    if not (os.path.exists(EMB_CACHE_FILE) and os.path.exists(EMB_META_FILE)):
        return False
    try:
        _cached_chunk_embs = torch.load(EMB_CACHE_FILE, map_location="cpu")
        with open(EMB_META_FILE, "r", encoding="utf-8") as f:
            _cached_chunk_ids = json.load(f)
        return True
    except Exception as e:
        print(f"缓存加载失败：{e}")
        _cached_chunk_embs, _cached_chunk_ids = None, None
        return False

In [24]:
##对用户问题通过DeepSeek-V3生成中文Task Description
import re
from typing import Optional

def generate_query_task(question: str, model_path: str = "") -> Optional[str]:
    """
    使用 DeepSeek-V3 API 生成中文任务描述（解题步骤）。
    输入 question 已经是中文子问题。
    """
    # 中文版提示词 - 经过优化以精准匹配知识库格式
    prompt = f"""你是工程领域知识检索专家。你的目标是将具体的子问题转换为标准化的“任务描述”（Task Description），该描述必须完美匹配我们技术知识库中使用的格式。

检索目标（知识库中的任务描述）使用严格的顺序格式："1. 确定 [输入1]。 2. 确定 [输入2]。 ... N. 计算 [公式]。 N+1. 输出 [结果]。"

【目标格式指南】
- **明确输入**：单独列出公式所需的每个变量。
- **LaTeX 数学符号**：对符号使用标准 LaTeX（例如 \\sigma_{{20}}, \\theta, \\times）。**严禁**使用 Markdown 代码块或 $ 分隔符。
- **逐步逻辑**：在“计算”步骤之前，逻辑性地列出变量获取步骤。
- **显式公式**：如果在“计算”步骤中可推断，请包含具体的公式表达式。

【高质量示例】

示例 A (无功补偿):
输入: "根据有功功率P和功率因数计算无功补偿容量 Q_c。"
输出: 1. 确定最小负荷有功功率 P_{{min}}。 2. 获取最小负荷功率因数正切值 \\tan\\phi_{{1min}}。 3. 获取目标功率因数正切值 \\tan\\phi_{{2}}。 4. 计算 Q_{{Cmin}} \\le P_{{min}} \\times (\\tan\\phi_{{1min}} - \\tan\\phi_{{2}})。 5. 输出最大允许 Q_{{Cmin}}。

示例 B (电导率修正):
输入: "将26度测量的电导率转换为标准20度电导率。"
输出: 1. 确定测量体积电导率 \\sigma_\\theta。 2. 确定温度系数 b。 3. 测量实验温度 \\theta。 4. 计算 \\sigma_{{20}} = \\sigma_\\theta \\times [1 - b(\\theta - 20)]。 5. 输出修正电导率 \\sigma_{{20}}。

示例 C (等值盐密):
输入: "根据盐度和清洗参数计算 ESDD。"
输出: 1. 确定盐度 S_s。 2. 测量清洗用水体积 V。 3. 测量表面积 A。 4. 计算 \\text{{ESDD}} = S_s \\times V / A。 5. 输出 \\text{{ESDD}}。

【你的任务】
为以下问题生成标准化的任务描述：
问题: "{question}"

要求:
1. **严格**遵循 "1. 确定 ... 2. 测量 ... 3. 计算 ... 4. 输出 ..." 的流程。
2. 逐步拆解公式所需的变量。
3. 对所有变量和公式使用原始 LaTeX 格式（例如 \\sigma_\\theta, \\times）。
4. **不要**使用 $ 或 $$ 等 Markdown 分隔符。直接书写 LaTeX。
5. 输出为包含编号步骤的**单一段落**。

任务描述:"""

    try:
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3",
            messages=[
                {"role": "system", "content": "你是一个将工程问题转化为可执行解题步骤的专家。"},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
            max_tokens=600,
        )
        generated_text = response.choices[0].message.content or ""

        # 简单的清理，去除可能的Markdown标记
        clean_text = generated_text.strip()
        clean_text = re.sub(r"^```.*?\n", "", clean_text)
        clean_text = re.sub(r"```$", "", clean_text)
        
        # 如果GPT输出了换行列表，尝试合并为单行
        if "\n" in clean_text:
             lines = [ln.strip() for ln in clean_text.split('\n') if ln.strip()]
             # 简单的启发式：如果看起来像列表，就用空格连接
             clean_text = " ".join(lines)
        
        return clean_text

    except Exception as e:
        print(f"DeepSeek-V3 生成解题步骤失败: {str(e)}")
        return None


In [25]:
def find_top_k_chunks(query_task: str, chunks: List[Dict], model, tokenizer, k: int = 3) -> Tuple[List[Dict], List[float]]:
    """
    使用缓存的chunk embeddings进行快速匹配，返回Top K个candidates及其分数。
    """
    global _cached_chunk_embs, _cached_chunk_ids, _similarity_log_buffer

    # 1. 尝试加载缓存或计算
    if _cached_chunk_embs is None or _cached_chunk_ids is None:
        if not _load_cached_embeddings():
            # 只有当磁盘没有缓存时，才进行预计算
            precompute_chunk_embeddings(chunks, model, tokenizer)

    try:
        # 对查询文本编码（先清理）
        # 【修改】使用新的 get_embeddings 逻辑（自动处理设备回退）
        # 建议这里仍显式请求 CPU 以最大限度避免 OOM，如果系统有空闲则 get_embeddings 会自动决定
        # 但鉴于目前的严重 OOM，我们默认先传 None 让他自己试，或者传 "cpu"
        # 为了保证绝对稳定，这里传 "cpu"
        query_task_clean = _strip_latex_markdown(query_task)
        # get_embeddings 现在具备自动回退能力，但这里直接指定 cpu 可跳过尝试
        query_emb = get_embeddings([query_task_clean], model, tokenizer, device="cpu")[0]  # [D]
        
        # 确保缓存也在CPU上以便计算
        if _cached_chunk_embs.device.type != 'cpu':
             _cached_chunk_embs = _cached_chunk_embs.cpu()
             
        scores = torch.matmul(_cached_chunk_embs, query_emb).tolist()  # [N]

        # 获取 Top K
        top_k_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        
        results = []
        result_scores = []
        
        idx_map = {str(c.get('chunk_id', f'idx_{i}')): i for i, c in enumerate(chunks)}

        _similarity_log_buffer.append(f"Top {k} 检索结果：")
        
        for rank, idx in enumerate(top_k_indices, 1):
            score = scores[idx]
            cid = _cached_chunk_ids[idx]
            
            # Find original chunk
            chunk_idx = idx_map.get(cid, idx)
            chunk = chunks[chunk_idx].copy() 
            
            chunk['_match_score'] = float(score)
            chunk['_match_chunk_id'] = str(cid)
            
            results.append(chunk)
            result_scores.append(score)
            
            log_line = f"  #{rank} [{cid}] sim={score:.4f}"
            print(log_line)
            _similarity_log_buffer.append(log_line)

        return results, result_scores

    except Exception as e:
        err = f"快速匹配出错：{e}, 回退到Top-1旧逻辑返回单元素的list"
        print(err)
        _similarity_log_buffer.append(err)
        return ([chunks[0]], [0.0]) if chunks else ([], [])

In [26]:
## 显示匹配到的公式信息（清理版）
def show_matched_formula_info(chunk: Dict):
    """显示匹配到的公式详细信息（对文本做LaTeX/Markdown清理）"""
    print("\n🎯 匹配到的公式信息：")
    print("="*60)
    
    # 显示核心信息
    if '标题' in chunk and chunk['标题']:
        print(f"📌 标题：{_strip_latex_markdown(chunk['标题'])}")
    if '公式' in chunk and chunk['公式']:
        print(f"📐 公式：{_strip_latex_markdown(chunk['公式'])}")
    if '关键词' in chunk and chunk['关键词']:
        keywords = chunk['关键词']
        if isinstance(keywords, list):
            print(f"🔑 关键词：{', '.join(_strip_latex_markdown(k) for k in keywords if k)}")
        else:
            print(f"🔑 关键词：{_strip_latex_markdown(str(keywords))}")
    
    # 显示变量定义
    if '变量定义' in chunk and chunk['变量定义']:
        print("\n📊 变量定义：")
        for var in chunk['变量定义']:
            if isinstance(var, dict):
                symbol = _strip_latex_markdown(var.get('符号', ''))
                name = _strip_latex_markdown(var.get('名称', ''))
                meaning = _strip_latex_markdown(var.get('物理意义', ''))
                unit = _strip_latex_markdown(var.get('单位', ''))
                note = _strip_latex_markdown(var.get('注', ''))
                print(f"  • {symbol}：{name}")
                if meaning:
                    print(f"    物理意义：{meaning}")
                if unit:
                    print(f"    单位：{unit}")
                if note:
                    print(f"    注：{note}")
    
    # 显示适用条件
    if '适用条件' in chunk and chunk['适用条件']:
        print(f"\n📋 适用条件：{_strip_latex_markdown(chunk['适用条件'])}")
    
    # 显示任务描述
    if 'task_description' in chunk and chunk['task_description']:
        print(f"\n🎯 解题流程：{_strip_latex_markdown(chunk['task_description'])}")
    
    print("="*60)

In [27]:
## 构建调用知识库之后强化Prompt（清理版）
def build_enhanced_prompt(user_question: str, query_task: str, chunk: Dict) -> str:
    """
    构造强化prompt，同时整合用户原始问题、查询task_description和chunk完整信息
    对所有文本进行清理，去除LaTeX/Markdown痕迹，保证可读性。
    """
    # 1. 格式化chunk信息（保留所有键值对，提升可读性）
    chunk_info = []
    for key, value in chunk.items():
        if not value:
            continue
        if isinstance(value, list):
            if key == "变量定义":
                var_details = []
                for v in value:
                    if isinstance(v, dict):
                        var_str = f"{_strip_latex_markdown(v.get('符号', ''))}：{_strip_latex_markdown(v.get('名称', ''))}（{_strip_latex_markdown(v.get('物理意义', ''))}，单位：{_strip_latex_markdown(v.get('单位', ''))}"
                        if '注' in v and v['注']:
                            var_str += f"，注：{_strip_latex_markdown(v['注'])}"
                        var_str += ")"
                        var_details.append(var_str)
                    else:
                        var_details.append(_strip_latex_markdown(str(v)))
                if var_details:
                    chunk_info.append(f"{key}：\n  - " + "\n  - ".join(var_details))
            elif key == "关键词":
                keywords_str = ', '.join(_strip_latex_markdown(str(item)) for item in value if item)
                if keywords_str:
                    chunk_info.append(f"{key}：{keywords_str}")
            else:
                string_items = []
                for item in value:
                    if isinstance(item, dict):
                        if '名称' in item:
                            string_items.append(_strip_latex_markdown(item['名称']))
                        elif '符号' in item:
                            string_items.append(_strip_latex_markdown(item['符号']))
                        else:
                            string_items.append(_strip_latex_markdown(str(item)))
                    else:
                        string_items.append(_strip_latex_markdown(str(item)))
                string_items = [item for item in string_items if item.strip()]
                if string_items:
                    chunk_info.append(f"{key}：\n  - " + "\n  - ".join(string_items))
        else:
            chunk_info.append(f"{key}：{_strip_latex_markdown(str(value))}")
    
    chunk_info_str = "\n".join(chunk_info)
    
    # 2. 构建最终prompt，明确关联原始问题和任务描述
    prompt = f"""请严格按照以下逻辑链回答问题：
(1). 先理解用户核心需求：
{_strip_latex_markdown(user_question)}

(2). 再明确任务执行步骤：
{_strip_latex_markdown(query_task)}

(3). 最后结合参考知识完成解答。

参考知识：
{chunk_info_str}

回答要求：
- 必须使用参考知识中的公式和变量定义
- 步骤需对应任务描述中的操作流程
- 最终结果需标注单位，与变量定义中的单位一致
- 禁止使用参考知识以外的公式或概念

请开始解答："""
    return prompt

In [28]:
## 构建综合Prompt工具函数
def build_comprehensive_prompt(
    user_question: str,
    sub_problems: List[str], 
    task_descriptions: List[str], 
    matched_chunks: List[Dict]
) -> str:
    """
    构造综合的强化prompt (严格适配中文知识库Key)
    """
    
    n = len(sub_problems)
    # 补齐
    if len(task_descriptions) < n:
        task_descriptions += [_strip_latex_markdown(sp) for sp in sub_problems[len(task_descriptions):]]
    if len(matched_chunks) < n:
        matched_chunks += [{} for _ in range(n - len(matched_chunks))]
    
    # 1. 构建子问题解析部分 (English Structure)
    sub_problems_section = []
    for i in range(n):
        sub_prob = sub_problems[i]
        task_desc = task_descriptions[i]
        sub_problems_section.append(f"""
【子问题 {i+1}】
问题描述：{_strip_latex_markdown(sub_prob)}
解题思路：{_strip_latex_markdown(task_desc)}""")
    
    sub_problems_str = "\n".join(sub_problems_section)
    
    # 2. 构建所有相关知识部分 (严格使用中文Key)
    all_knowledge = []
    for i in range(n):
        chunk = matched_chunks[i] if i < len(matched_chunks) else {}
        chunk_info = []
        
        if isinstance(chunk, dict):
            # (1) chunk_id
            if 'chunk_id' in chunk:
                chunk_info.append(f"chunk_id: {chunk['chunk_id']}")

            # (2) 变量定义
            if '变量定义' in chunk and isinstance(chunk['变量定义'], list):
                vars_list = []
                for v in chunk['变量定义']:
                    if isinstance(v, dict):
                        # 严格使用中文Key: 符号, 名称, 物理意义, 单位
                        s = f"{v.get('符号','')} : {v.get('名称','')} (含义: {v.get('物理意义','')}, 单位: {v.get('单位','')})"
                        vars_list.append(s)
                if vars_list:
                    chunk_info.append(f"变量定义:\n    - " + "\n    - ".join(vars_list))
            
            # (3) 其他字段
            # task_description
            if 'task_description' in chunk and chunk['task_description']:
                 chunk_info.append(f"task_description: {_strip_latex_markdown(str(chunk['task_description']))}")
            
            # 中文Key字段
            for key in ['公式', '关键词', '完整上下文', '潜在问题', '所属章节', '难度级别', '学科领域']:
                if key in chunk and chunk[key]:
                    val = chunk[key]
                    if isinstance(val, list):
                        val = ", ".join(str(x) for x in val)
                    chunk_info.append(f"{key}: {_strip_latex_markdown(str(val))}")
        
        knowledge_block = f"""
【知识块 {i+1}】 (对应子问题 {i+1})
{chr(10).join(chunk_info)}"""
        all_knowledge.append(knowledge_block)
    
    all_knowledge_str = "\n".join(all_knowledge)
    
    # 3. 构建最终的综合prompt (Chinese Structure)
    comprehensive_prompt = f"""请严格按照以下逻辑链解决这个复合工程问题：

.
【原始复合问题】
{_strip_latex_markdown(user_question)}

.
【问题拆解分析】
该问题已被拆解为 {n} 个相关的子问题：
{sub_problems_str}

.
【参考知识库】
{all_knowledge_str}

.
【解答要求】
1. 必须按照子问题的逻辑顺序依次求解。
2. 每个子问题仅使用参考知识库中对应的公式和变量定义。
3. 前一个子问题的输出应作为下一个子问题的输入。
4. 最终结果必须包含与定义一致的单位。
5. 提供完整的推导过程和最终答案。
6. 在最终答案中，所有变量和公式请使用 LaTeX 格式 (例如 \\sigma_\\theta, \\times, \\sqrt{{...}})。

开始解答："""
    return comprehensive_prompt

# Agent 多角色多轮交互（集成到本RAG）

目标：在现有 RAG 流程上引入“规划-检索-求解-评审-编辑”的多角色、多轮闭环，以提高答案质量与稳健性。

核心角色：
- 规划者 Planner：将复合问题拆解为可执行的子问题。
- 检索者 Retriever：为每个子问题生成 task_description，并检索最相关的知识块。
- 求解者 Solver：基于综合 Prompt 生成解答。
- 评审者 Critic：对解答进行一致性/约束/完整性审查并打分。
- 编辑者 Editor：根据评审意见调整策略（如限制两条公式、重排步骤），进入下一轮。

两种常见模式：
- 多步复合模式（默认）：允许任意步数的子问题，追求完整链路。
- 双公式模式（Two-Formula）：强制仅使用2个子问题/2条公式，突出“链式单题”要求。

使用方式：运行到文档末尾的演示单元格，设置问题与模式，系统将自动进行2-3轮交互并输出最终答案、Prompt和日志。

In [29]:
# Agent 配置
import os, time, json

AGENT_CFG = {
    "max_rounds": 3,                 # 最多迭代轮数
    "two_formula_mode": False,       # True 则强制仅两条公式/两步子问题
    "min_score": 0.85,               # 评审者通过阈值（0-1），0.85及以上通过
    "min_similarity": 0.75,          # 子问题匹配相似度阈值
    "save": {
        "prompt_path": "full_prompt_agent/agent_prompt5.txt",
        "log_dir": "logs"
    }
}

def _ensure_dirs():
    os.makedirs(os.path.dirname(AGENT_CFG["save"]["prompt_path"]), exist_ok=True)
    os.makedirs(AGENT_CFG["save"]["log_dir"], exist_ok=True)

def _now_tag():
    return time.strftime("%Y%m%d-%H%M%S", time.localtime())

In [30]:
# 角色实现：Planner / Retriever / Solver / Critic / Editor
import re, json
from typing import Tuple, List, Dict, Any

def role_planner(user_question: str, two_formula_mode: bool=False) -> List[str]:
    # 复用已有拆解函数；双公式模式下仅保留前2个子问题
    subs = decompose_complex_problem(user_question) or [user_question]
    if two_formula_mode and len(subs) > 2:
        subs = subs[:2]
    return subs

def _parse_json_loose(s: str) -> Dict[str, Any]:
    try:
        return json.loads(s)
    except Exception:
        s2 = re.sub(r"^```(json)?|```$", "", s.strip(), flags=re.MULTILINE)
        try:
            return json.loads(s2)
        except Exception:
            return {}

def select_best_chunk_with_gpt(sub_prob: str, candidates: List[Dict]) -> Dict:
    """
    使用DeepSeek-V3从Top-3候选中筛选最合适的一个。
    目的是过滤掉向量相似度高但物理意义错误（干扰项）的chunk。
    """
    if not candidates:
        return {}
    if len(candidates) == 1:
        return candidates[0]

    cands_text = []
    for i, c in enumerate(candidates, 1):
        # 格式化候选信息 - 包含完整上下文
        info = f" 【候选 {i}】 (相似度: {c.get('_match_score', 0):.4f})\n"
        
        # 1. 公式
        if c.get('公式'): 
            info += f"   - 公式: {_strip_latex_markdown(c['公式'])}\n"
        elif c.get('formula'):
            info += f"   - Formula: {_strip_latex_markdown(c['formula'])}\n"
            
        # 2. 任务/标题
        desc = c.get('task_description') or c.get('标题') or ""
        if desc: 
            info += f"   - 任务/标题: {_strip_latex_markdown(desc)}\n"
        
        # 3. 变量定义
        if c.get('变量定义'): 
            vars_def = c['变量定义']
            if isinstance(vars_def, list):
                 vars_str = ", ".join([str(v) for v in vars_def])
            else:
                 vars_str = str(vars_def)
            info += f"   - 变量定义: {vars_str}\n"
        elif c.get('variable_definitions'):
            vars_def = c['variable_definitions']
            if isinstance(vars_def, list):
                 vars_str = ", ".join([str(v) for v in vars_def])
            else:
                 vars_str = str(vars_def)
            info += f"   - Variable Definitions: {vars_str}\n"

        # 4. 关键词
        if c.get('关键词') or c.get('keywords'):
             info += f"   - 关键词: {c.get('关键词') or c.get('keywords')}\n"
            
        # 5. 完整上下文
        context = c.get('完整上下文') or c.get('full_context') or ""
        if context:
            info += f"   - 完整上下文: {_strip_latex_markdown(context)}\n"
            
        cands_text.append(info)
        
    cands_str = "\n".join(cands_text)
    
    prompt = f"""
你是一个高效的工程领域助手。你的目标是从提供的候选项中为特定的计算步骤筛选出**最合适的一个**知识块。

【待解决的当前子问题】
"{sub_prob}"

【候选知识块 (向量检索 Top 3)】
{cands_str}

【筛选标准】
1. **数学相关性**：候选项中的“公式”是否**直接**解决了该子问题？
2. **变量匹配**：子问题中所需的输入是否与知识块产生/需要的变量匹配？
3. **情境检查**：物理情境（例如“清洗测试”、“温度修正”）是否正确？
4. **排除干扰项**：拒绝那些向量相似度高但物理意义错误的候选项。

【你的任务】
输出一个 JSON 对象，指示最佳索引（从1开始）。
格式：
{{
    "selected_index": 1, 
    "confidence": "High/Medium/Low",
    "reason": "逐步解释为什么候选 N 是最匹配的，以及为什么拒绝其他候选项。"
}}

Output JSON:
"""
    try:
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=1000
        )
        content = response.choices[0].message.content
        data = _parse_json_loose(content)
        
        # 增强鲁棒性：处理可能返回 None 或 非数字 的情况
        idx_val = data.get("selected_index")
        if idx_val is None:
             idx_val = 1
        try:
             idx = int(idx_val) - 1
        except ValueError:
             idx = 0
             
        if 0 <= idx < len(candidates):
            selected = candidates[idx]
            # print(f"✅ GPT Selected Candidate #{idx+1}")
            return selected
    except Exception as e:
        print(f"❌ GPT Selection failed: {e}. Defaulting to Top-1.")
    
    return candidates[0]

def role_retriever(sub_problems: List[str]) -> Tuple[List[str], List[Dict], List[float]]:
    """
    修改后的Retriever:
    1. 为每个子问题检索Top-3
    2. 使用DeepSeek-V3筛选最佳的1个
    3. 返回选中chunk以及 Top-1 Similarity Scores (用于Threshold Check)
    """
    task_descs, matched = [], []
    top1_scores = []  # 用于 "top1 similarity" 检查
    
    for sp in sub_problems:
        t = generate_query_task(sp) or _strip_latex_markdown(sp)
        task_descs.append(_strip_latex_markdown(t))
        
        # 1. Get Top 3 (Retrieval)
        cands, scores = find_top_k_chunks(task_descs[-1], all_chunks, model, tokenizer, k=3)
        
        # 记录每组这一轮里 Top 1 的分数
        top1_scores.append(scores[0] if scores else 0.0)
        
        # 2. GPT Select
        best = select_best_chunk_with_gpt(sp, cands)
        matched.append(best)
        
    return task_descs, matched, top1_scores

def role_solver(user_question: str, sub_problems: List[str], task_descs: List[str], matched_chunks: List[Dict], two_formula_mode: bool=False) -> Tuple[str, str]:
    # 构建综合prompt + Agent约束
    base_prompt = build_comprehensive_prompt(user_question, sub_problems, task_descs, matched_chunks)
    constraints = []
    if two_formula_mode:
        constraints.append("必须严格使用不超过两条公式，优先使用已检索到的两个知识块对应公式；若不满足，请明确指出不足之处并停止延伸。")
    constraints.append("所有推导步骤需引用知识块中的变量定义与单位，不得引入未知公式。")
    final_prompt = base_prompt + "\n【Agent约束】\n- " + "\n- ".join(constraints)
    # 使用GPT求解
    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3",
        messages=[
            {"role": "system", "content": "你是严谨的工程领域解题专家，严格遵守用户提供的知识与约束。"},
            {"role": "user", "content": final_prompt}
        ],
        temperature=0.2, max_tokens=1800,
    )
    answer = response.choices[0].message.content or ""
    return final_prompt, answer

def role_critic(user_question: str, matched_chunks: List[Dict], answer: str, two_formula_mode: bool=False) -> Dict[str, Any]:
    # 构造评审提示，要求输出JSON
    kb_brief = []
    for i, ch in enumerate(matched_chunks, 1):
        # 适配中文 Key
        title = ch.get('标题') or ch.get('task_description') or ''
        formula = ch.get('公式') or ch.get('formula') or ''
        kb_brief.append(f"[知识块{i}] 标题: {title}, 公式: {formula}")
        
    critic_prompt = f"""
你是严格的技术评审员，请针对下面解答做一致性与约束评估，并输出JSON：

【原始问题】
{_strip_latex_markdown(user_question)}

【使用知识】
{chr(10).join(_strip_latex_markdown(x) for x in kb_brief)}

【模型解答】
{_strip_latex_markdown(answer)}

【评审要点】
- 一致性：是否仅使用知识块中的定义/公式/单位
- 完整性：步骤是否闭环、量纲与单位是否正确
- 约束：{'仅允许两条公式' if two_formula_mode else '允许多步'}

【输出JSON字段（严格JSON，不要额外文本）】
{{
  "score": 0.0-1.0,
  "needs_revision": true/false,
  "issues": ["问题1", "问题2"],
  "suggestions": ["建议1", "建议2"]
}}
"""
    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3",
        messages=[
            {"role": "system", "content": "你是严格的技术评审员，输出必须是JSON。"},
            {"role": "user", "content": critic_prompt}
        ],
        temperature=0.0, max_tokens=600,
    )
    raw = response.choices[0].message.content or ""
    data = _parse_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    data.setdefault("score", 0.0)
    data.setdefault("needs_revision", True)
    if "issues" not in data:
        data["issues"] = ["未能解析模型解答或JSON不合法"]
    data.setdefault("suggestions", [])
    return data

def role_editor(sub_problems: List[str], feedback: Dict[str, Any], cfg: Dict[str, Any]) -> List[str]:
    sp = list(sub_problems)
    if cfg.get("two_formula_mode", False):
        sp = sp[:2] if len(sp) >= 2 else sp
    if feedback.get("needs_revision", True):
        # 简单策略
        pass
    return sp

In [31]:
# 编排器：多轮对话式Agent主流程
from typing import NamedTuple

class AgentResult(NamedTuple):
    answer: str
    final_prompt: str
    score: float
    rounds: int
    feedbacks: List[Dict[str, Any]]

def run_agent(user_question: str, cfg: Dict[str, Any]) -> AgentResult:
    # 0. 自动加载必要的全局资源 (模型 & 知识库)
    global model, tokenizer, all_chunks
    
    if 'model' not in globals() or 'tokenizer' not in globals():
        print("🔧 检测到模型未加载，正在初始化 Embedding 模型...")
        model, tokenizer = load_embedding_components()
        print("✅ 模型加载完成")

    if 'all_chunks' not in globals() or not all_chunks:
        print("📊 检测到知识库未加载，正在加载默认路径数据...")
        # 尝试使用默认路径
        default_path = "/data1/oysl/RAG/jsons_EE_Chi" # Chinese Path (Default)
        if not os.path.exists(default_path):
             default_path = "/data1/oysl/RAG/jsons_EE_Eng" # Fallback if needed
        all_chunks = load_jsonl_files(default_path)

    _ensure_dirs()
    logs = []
    feedbacks: List[Dict[str, Any]] = []
    two_formula_mode = cfg.get("two_formula_mode", False)
    max_rounds = int(cfg.get("max_rounds", 2))
    min_score = float(cfg.get("min_score", 0.8))
    sim_threshold = float(cfg.get("min_similarity", 0.75))
    
    # 最佳结果容器 - 基于“Round Metric”选择
    # Metric = Average Score of SELECTED chunks
    best_similarity_metric = -1.0 
    best_results = {
        "round": 0,
        "score": 0.0,
        "answer": "",
        "prompt": "",
        "feedback": {}
    }
    
    sub_problems: List[str] = []
    
    rd = 1
    while rd <= max_rounds:
        logs.append(f"=== Round {rd} ===")
        print(f"\n🟢 开始第 {rd} 轮 ...")
        
        # 1) Planner
        if rd == 1:
            sub_problems = role_planner(user_question, two_formula_mode=two_formula_mode)
        else:
            pass
            
        logs.append(f"Planner: 子问题数量={len(sub_problems)}")
        
        # 2) Retriever
        # 返回 GPT 筛选后的 'matched' (list of chunks) 和 'top1_scores' (list of Top-1 Sim)
        task_descs, matched, top1_scores = role_retriever(sub_problems)
        
        # 计算本轮的两个关键指标
        selected_scores = [c.get('_match_score', 0.0) for c in matched]
        
        # A. Threshold Check (Top-1)
        if any(s < sim_threshold for s in top1_scores):
            msg = f"⚠️ 本轮Top-1相似度过低 (Threshold={sim_threshold:.2f}). Top-1 Scores: {[f'{s:.2f}' for s in top1_scores]}. 放弃本轮结果，尝试调整..."
            print(msg)
            logs.append(msg)
            
            mock_feedback = {
                 "needs_revision": True,
                 "issues": ["Retrieval similarity too low. Rephrase sub-problems."]
            }
            logs.append("✏️ 因相似度不足触发 Editor 修改计划...")
            sub_problems = role_editor(sub_problems, mock_feedback, cfg)
            rd += 1
            continue  
            
        # B. Round Selection Metric
        current_round_sim_metric = sum(selected_scores) / len(selected_scores) if selected_scores else 0.0

        logs.append("Retriever: Matched={} | Selected Sims={} | Round Metric={:.4f}".format(
            len(matched), [f"{s:.4f}" for s in selected_scores], current_round_sim_metric
        ))
        
        # 4) Solver
        final_prompt, answer = role_solver(user_question, sub_problems, task_descs, matched, two_formula_mode=two_formula_mode)
        logs.append(f"Solver: 生成答案长度={len(answer)}")
        
        # 5) Critic
        fb = role_critic(user_question, matched, answer, two_formula_mode=two_formula_mode)
        feedbacks.append(fb)
        score = float(fb.get("score", 0.0))
        logs.append(f"Critic: score={score}, needs_revision={fb.get('needs_revision', True)}")
        print(f"📊 本轮 Critic 打分: {score} | Sim Metric: {current_round_sim_metric:.4f}")

        # 筛选逻辑：保留“最合适的一轮”
        if current_round_sim_metric > best_similarity_metric:
            best_similarity_metric = current_round_sim_metric
            best_results = {
                "round": rd,
                "score": score,
                "answer": answer,
                "prompt": final_prompt,
                "feedback": fb
            }
            print(f"🌟此轮 Metric ({current_round_sim_metric:.4f}) 为当前最高，暂定为最佳轮次。")
            
        # 6) Stop or Edit
        if score >= min_score and not fb.get("needs_revision", True):
            logs.append("⏹ 满足阈值 (Critic). Stopping.")
            print("✅ 达到 Critic 分数阈值，提前结束循环。")
            break
        else:
            logs.append("✏️ 阈值未达. Editing...")
            sub_problems = role_editor(sub_problems, fb, cfg)
            rd += 1
    
    # 最终结果输出
    if best_results["round"] > 0:
        chosen_answer = best_results["answer"]
        chosen_prompt = best_results["prompt"]
        chosen_feedback = best_results["feedback"]
        print(f"\n🏆 最终选择第 {best_results['round']} 轮结果 (Sim Metric: {best_similarity_metric:.4f}, Critic Score: {best_results['score']})")
    else:
        # 兜底
        chosen_answer = answer
        chosen_prompt = final_prompt
        chosen_feedback = feedbacks[-1] if feedbacks else {}
        print("\n⚠️ 未能找到满意的结果，返回最后一次尝试。")
    
    logs.append(f"Best Round: {best_results['round']} (Sim: {best_similarity_metric:.4f})")
    
    ts = _now_tag()
    log_path = os.path.join(cfg["save"]["log_dir"], f"agent_run_{ts}.txt")
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("\n".join(logs))
        f.write("\n\nBest Round Feedback:\n")
        try:
            f.write(json.dumps(chosen_feedback, ensure_ascii=False, indent=2))
        except Exception:
            f.write(str(chosen_feedback))
            
    with open(cfg["save"]["prompt_path"], "w", encoding="utf-8") as f:
        f.write(chosen_prompt)
        
    print(f"Log saved: {log_path}")
    print(f"Prompt saved: {cfg['save']['prompt_path']}")
    return AgentResult(answer=chosen_answer, final_prompt=chosen_prompt, score=best_results["score"], rounds=len(feedbacks), feedbacks=feedbacks)

In [32]:
# ================= 批量运行模式 =================
# 配置区域
BATCH_INPUT_FILE = "/data1/oysl/RAG/EE_Question/EE_Question_Batch.txt"  # 问题输入文件 (中文)
OUTPUT_DIR = "Prompt_chi"       # 结果输出文件夹 (相对于当前目录)

# ***在此处设置处理范围***
START_INDEX = 5        # 起始编号 (例如 1 或 101)
END_INDEX = 5        # 结束编号 (例如 100 或 200)

# 更新Agent全局默认配置
AGENT_CFG["two_formula_mode"] = False       
AGENT_CFG["save"]["log_dir"] = "logs"

import os
import traceback
import gc

# 【修复】强制释放旧 Model 全局对象，确保 run_agent 加载新的 CPU Safe Model
if 'model' in globals():
    try:
        del model
        print("🧹 Cleaned up old global 'model' object.")
    except:
        pass
if 'tokenizer' in globals():
    try:
        del tokenizer
        print("🧹 Cleaned up old global 'tokenizer' object.")
    except:
        pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# 确保输出目录存在
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. 读取问题
questions = []
if os.path.exists(BATCH_INPUT_FILE):
    with open(BATCH_INPUT_FILE, "r", encoding="utf-8") as f:
        # 简单处理：过滤掉注释行和空行
        for line in f:
            ln = line.strip()
            if ln and not ln.startswith("#"):
                questions.append(ln)
    print(f"📖 从 {BATCH_INPUT_FILE} 加载了 {len(questions)} 个问题。")
else:
    print(f"⚠️ 未找到文件 {BATCH_INPUT_FILE}，请先在 {BATCH_INPUT_FILE} 中填入问题。")
    # questions = [] # Assuming user will create it

# 2. 批量执行
# 切片获取指定范围的问题. Python list索引从0开始, 所以要减1
target_questions = questions[START_INDEX-1 : END_INDEX]

if not target_questions:
    print(f"⚠️ 警告: 指定范围 {START_INDEX}-{END_INDEX} 没有取到问题 (总问题数: {len(questions)})")
else:
    print(f"🎯 准备处理序号 {START_INDEX} 到 {END_INDEX} 的问题，共 {len(target_questions)} 个。")

    for i, q in enumerate(target_questions):
        current_idx = START_INDEX + i
        
        # 构造输出文件名: Question101.txt ... 
        filename = f"Question{current_idx}.txt"
        output_path = os.path.join(OUTPUT_DIR, filename)
        
        print(f"\n" + "="*50)
        print(f"🚀 [Processing Item {current_idx}] (Batch Progress: {i+1}/{len(target_questions)})")
        print(f"📝 Question: {q[:50]}...")
        
        # 动态更新保存路径
        AGENT_CFG["save"]["prompt_path"] = output_path
        
        try:
            # 执行Agent
            res = run_agent(q, AGENT_CFG)
            print(f"✅ Item {current_idx} Finished. Score: {res.score}. Output -> {output_path}")
        except Exception as e:
            print(f"❌ Item {current_idx} Failed: {str(e)}")
            traceback.print_exc()
            # 遇到错误继续下一个
            continue

    print(f"\n🎉 范围 {START_INDEX}-{END_INDEX} 批量任务完成！结果保存在 {OUTPUT_DIR}/ 目录下。")

🧹 Cleaned up old global 'model' object.
🧹 Cleaned up old global 'tokenizer' object.
📖 从 /data1/oysl/RAG/EE_Question/EE_Question_Batch.txt 加载了 288 个问题。
🎯 准备处理序号 5 到 5 的问题，共 1 个。

🚀 [Processing Item 5] (Batch Progress: 1/1)
📝 Question: 5. 某盘形悬式绝缘子，上表面积A_t=500cm²，测得ESDD_t=0.04 mg/cm²；下表...
🔧 检测到模型未加载，正在初始化 Embedding 模型...
🔄 正在初始化模型 (强制 CPU 模式)...
✅ 模型加载完成

🟢 开始第 1 轮 ...
🔧 正在调用DeepSeek-V3拆解复合题目...
✅ DeepSeek-V3拆解完成
📋 拆解结果（原始）：
【变量提取】
已知变量: 
- 上表面积 A_t = 500 cm²
- 上表面等值盐密 ESDD_t = 0.04 mg/cm²
- 下表面积 A_b = 700 cm²
- 下表面等值盐密 ESDD_b = 0.10 mg/cm²
- 蒸馏水体积 V = 600 cm³
- 盐度 S_s = 0.12 kg/m³

目标变量: 
- 平均等值盐密 \overline{ESDD}

【公式路径分析】
1. 计算上表面盐量 M_t = ESDD_t \times A_t
2. 计算下表面盐量 M_b = ESDD_b \times A_b
3. 计算总盐量 M_{total} = M_t + M_b
4. 计算总表面积 A_{total} = A_t + A_b
5. 计算平均等值盐密 \overline{ESDD} = M_{total} / A_{total}

【子问题拆解】

【子问题1】计算上表面盐量 M_t
→ 输入: ESDD_t = 0.04 mg/cm², A_t = 500 cm²
→ 公式: M_t = ESDD_t \times A_t
→ 输出: M_t = 0.04 \times 500 = 20 mg

【子问题2】计算下表面盐量 M_b
→ 输入: ESDD_b = 0.10 mg/cm², A_

Traceback (most recent call last):
  File "/tmp/ipykernel_2351564/2786592718.py", line 77, in <module>
    res = run_agent(q, AGENT_CFG)
  File "/tmp/ipykernel_2351564/2009190520.py", line 132, in run_agent
    chosen_answer = answer
                    ^^^^^^
UnboundLocalError: cannot access local variable 'answer' where it is not associated with a value
